# Hiring Compass DB Overview

Purpose: provide a quick, global overview of the current DB state.

How to use:
1. Choose a DB path below (dev/prod/smoke test).
2. Run all cells top to bottom.


In [4]:
import sqlite3
from pathlib import Path

import pandas as pd

# Choose your DB here:
# - Dev:   ./run/dev/data/local/state.sqlite
# - Prod:  ./run/prod/data/state.sqlite
# - Test:  ./run/prod/data/state.test.sqlite
DB_PATH = Path("../run/dev/data/local/state.sqlite").resolve()
DB_PATH, DB_PATH.exists()


(PosixPath('/Users/rubenchekroun/Code/Scripts perso/hiring_compass_au/run/dev/data/local/state.sqlite'),
 True)

## 0) Table row counts
High-level size overview for core tables.


In [5]:
with sqlite3.connect(DB_PATH) as conn:
    df_counts = pd.read_sql_query(
        """
        SELECT "job_ads" AS table_name, COUNT(*) AS n FROM job_ads
        UNION ALL
        SELECT "job_ad_enrichment", COUNT(*) FROM job_ad_enrichment
        UNION ALL
        SELECT "seek_enrichment", COUNT(*) FROM seek_enrichment
        UNION ALL
        SELECT "company", COUNT(*) FROM company
        UNION ALL
        SELECT "email_job_hits", COUNT(*) FROM email_job_hits
        UNION ALL
        SELECT "email_job_ads", COUNT(*) FROM email_job_ads
        UNION ALL
        SELECT "emails", COUNT(*) FROM emails
        """,
        conn,
    )
df_counts


,table_name,n
0,job_ads,1090
1,job_ad_enrichment,2180
2,seek_enrichment,10
3,company,10
4,email_job_hits,4272
5,email_job_ads,0
6,emails,200


## 1) Enrichment status by type
Counts of `enrich_status` grouped by `enrich_type`.


In [6]:
with sqlite3.connect(DB_PATH) as conn:
    df_status = pd.read_sql_query(
        """
        SELECT enrich_type, enrich_status, COUNT(*) AS n
        FROM job_ad_enrichment
        GROUP BY enrich_type, enrich_status
        ORDER BY enrich_type ASC, enrich_status ASC
        """,
        conn,
    )
df_status


,enrich_type,enrich_status,n
0,jobDetails,error,54
1,jobDetails,ok,10
2,jobDetails,pending,1026
3,matchedSkills,pending,1090


## 2) Recent attempts
Check the latest attempts (should include the last smoke test run).


In [7]:
with sqlite3.connect(DB_PATH) as conn:
    df_recent = pd.read_sql_query(
        """
        SELECT job_id, enrich_type, enrich_status, http_status, last_attempt_at
        FROM job_ad_enrichment
        WHERE last_attempt_at IS NOT NULL
        ORDER BY last_attempt_at DESC
        LIMIT 20
        """,
        conn,
    )
df_recent


,job_id,enrich_type,enrich_status,http_status,last_attempt_at
0,4122,jobDetails,ok,200.0,2026-03-13T09:56:07+00:00
1,4117,jobDetails,ok,200.0,2026-03-13T09:56:06+00:00
2,4114,jobDetails,ok,200.0,2026-03-13T09:56:04+00:00
3,4111,jobDetails,ok,200.0,2026-03-13T09:56:03+00:00
4,4109,jobDetails,ok,200.0,2026-03-13T09:56:01+00:00
5,4108,jobDetails,ok,200.0,2026-03-13T09:56:00+00:00
6,4106,jobDetails,ok,200.0,2026-03-13T09:55:58+00:00
7,4102,jobDetails,ok,200.0,2026-03-13T09:55:57+00:00
8,4101,jobDetails,ok,200.0,2026-03-13T09:55:56+00:00
9,4100,jobDetails,ok,200.0,2026-03-13T09:55:54+00:00


## 3) Inspect a few updated rows
If you want to inspect a specific job_id, update the query below.


In [8]:
# Example: replace 123 with a real job_id from df_recent
JOB_ID = None  # e.g. 123

if JOB_ID is not None:
    with sqlite3.connect(DB_PATH) as conn:
        df_job = pd.read_sql_query(
            """
            SELECT *
            FROM job_ad_enrichment
            WHERE job_id = ?
            ORDER BY enrich_type
            """,
            conn,
            params=(JOB_ID,),
        )
    df_job


## 4) Recent job_ads updates
Latest rows by `last_seen_at` / `first_seen_at` when available.


In [9]:
with sqlite3.connect(DB_PATH) as conn:
    df_jobs = pd.read_sql_query(
        """
        SELECT id, source, external_job_id, title, company,
               last_seen_at, first_seen_at, listing_date_utc
        FROM job_ads
        ORDER BY
          CASE WHEN last_seen_at IS NULL THEN 1 ELSE 0 END ASC,
          last_seen_at DESC,
          id DESC
        LIMIT 20
        """,
        conn,
    )
df_jobs


,id,source,external_job_id,title,company,last_seen_at,first_seen_at,listing_date_utc
0,4122,seek,90640826,Data Analyst,Reo Group,2026-03-13T09:56:07+00:00,2026-03-03T19:15:00+00:00,2026-03-02T00:01:19.360Z
1,4117,seek,90650690,Digital Reporting and Systems Analyst,Corporate Carbon Advisory Pty Ltd,2026-03-13T09:56:06+00:00,2026-03-03T19:15:00+00:00,2026-03-02T04:37:17.643Z
2,4114,seek,90639355,Cloud Infrastructure Engineer,Nexia Australia,2026-03-13T09:56:04+00:00,2026-03-03T19:15:00+00:00,2026-03-01T22:32:10.572Z
3,4111,seek,90655712,Data Modeler,Peoplebank Australia NSW,2026-03-13T09:56:03+00:00,2026-03-03T19:15:00+00:00,2026-03-02T07:06:04.443Z
4,4109,seek,90646732,Data Modeller/Database Specialist,Randstad Digital,2026-03-13T09:56:01+00:00,2026-03-03T19:15:00+00:00,2026-03-02T03:03:36.450Z
5,4108,seek,90651924,Senior Databricks Engineer (Sydney Based),UpperGround by Hudson,2026-03-13T09:56:00+00:00,2026-03-03T19:15:00+00:00,2026-03-02T05:16:06.584Z
6,4106,seek,90652140,Lead Data Engineer,The University of Queensland,2026-03-13T09:55:58+00:00,2026-03-03T19:15:00+00:00,2026-03-02T05:22:23.100Z
7,4102,seek,90638862,Senior Data Engineer,"Talent – Specialists in tech, transformation &...",2026-03-13T09:55:57+00:00,2026-03-03T19:15:00+00:00,2026-03-01T21:53:05.092Z
8,4101,seek,90643765,Data Engineer - JDE - Contract,PRA,2026-03-13T09:55:56+00:00,2026-03-03T19:15:00+00:00,2026-03-02T01:48:39.225Z
9,4100,seek,90643477,Data Engineer,FinXL IT Professional Services,2026-03-13T09:55:54+00:00,2026-03-03T19:15:00+00:00,2026-03-02T01:40:19.102Z


## 5) Recent seek_enrichment updates
Latest rows by `job_id` (no timestamps in this table).


In [10]:
with sqlite3.connect(DB_PATH) as conn:
    df_seek = pd.read_sql_query(
        """
        SELECT job_id, advertiser_id, seo_normalised_role_title,
               work_types, insights_count, status
        FROM seek_enrichment
        ORDER BY job_id DESC
        LIMIT 20
        """,
        conn,
    )
df_seek


,job_id,advertiser_id,seo_normalised_role_title,work_types,insights_count,status
0,4122,25424904,Data Analyst,Full time,491.0,Expired
1,4117,44094174,Systems Analyst,Full time,617.0,Active
2,4114,2931,Cloud Infrastructure Engineer,Contract/Temp,122.0,Active
3,4111,28935911,Data Modeller,Contract/Temp,NaN,Active
4,4109,26537413,Database Specialist,Full time,NaN,Active
5,4108,41306130,Engineer,Full time,41.0,Active
6,4106,26331750,Data Engineer,Full time,NaN,Active
7,4102,23607501,Data Engineer,Full time,64.0,Active
8,4101,20101884,Data Engineer,Contract/Temp,71.0,Active
9,4100,45561458,Data Engineer,Contract/Temp,75.0,Active


## 6) Recent company updates
Company table has no timestamps; show latest ids.


In [11]:
with sqlite3.connect(DB_PATH) as conn:
    df_company = pd.read_sql_query(
        """
        SELECT id, name, industry, size, seek_company_id,
               seek_rating_value, seek_review_count
        FROM company
        ORDER BY id DESC
        LIMIT 20
        """,
        conn,
    )
df_company


,id,name,industry,size,seek_company_id,seek_rating_value,seek_review_count
0,10,Reo Group,NaN,NaN,25424904,NaN,NaN
1,9,Corporate Carbon Advisory Pty Ltd,NaN,NaN,44094174,NaN,NaN
2,8,Nexia Australia,Accounting,"101-1,000 employees",2931,3.6,20.0
3,7,Peoplebank Australia NSW,NaN,NaN,28935911,NaN,NaN
4,6,Randstad Digital,NaN,NaN,26537413,NaN,NaN
5,5,UpperGround by Hudson,NaN,NaN,41306130,NaN,NaN
6,4,The University of Queensland,Education & Training,"5,001-10,000 employees",26331750,3.7,203.0
7,3,"Talent – Specialists in tech, transformation &...",NaN,NaN,23607501,NaN,NaN
8,2,PRA,NaN,NaN,20101884,NaN,NaN
9,1,FinXL IT Professional Services,NaN,NaN,45561458,NaN,NaN
